In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
# @title
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

def plot_learning_curves(history, model_name="Моделі"):
    """Функція для малювання графіків Loss та Accuracy"""
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))

    # Графік 1: Втрати (Loss)
    ax[0].plot(history.history['loss'], label='Train Loss', color='blue', linewidth=2)
    ax[0].plot(history.history['val_loss'], label='Validation Loss', color='red', linewidth=2)
    ax[0].set_title(f'Крива навчання: Втрати (Loss) [{model_name}]', fontsize=14)
    ax[0].set_xlabel('Епохи', fontsize=12)
    ax[0].set_ylabel('Loss (Помилка)', fontsize=12)
    ax[0].legend(fontsize=12)
    ax[0].grid(True, linestyle='--', alpha=0.7)

    # Графік 2: Точність (Accuracy)
    ax[1].plot(history.history['accuracy'], label='Train Accuracy', color='blue', linewidth=2)
    ax[1].plot(history.history['val_accuracy'], label='Validation Accuracy', color='red', linewidth=2)
    ax[1].set_title(f'Крива навчання: Точність (Accuracy) [{model_name}]', fontsize=14)
    ax[1].set_xlabel('Епохи', fontsize=12)
    ax[1].set_ylabel('Accuracy', fontsize=12)
    ax[1].legend(fontsize=12)
    ax[1].grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.show()

In [ ]:
files = ["46343_30_dataset.csv", "759667_30_dataset.csv",
         "781756_30_dataset.csv", "844359_30_dataset.csv",
         "1066528_30_dataset.csv", "1360686_30_dataset.csv",
         "1449548_30_dataset.csv", "1455390_30_dataset.csv",
         "1818471_30_dataset.csv", "2598705_30_dataset.csv",
         "2638030_30_dataset.csv", "3509524_30_dataset.csv",
         "3997827_30_dataset.csv", "4018081_30_dataset.csv",
         "4314139_30_dataset.csv", "4426783_30_dataset.csv",
         "5132496_30_dataset.csv", "5383425_30_dataset.csv",
         "5498603_30_dataset.csv", "5797046_30_dataset.csv",
         "6220552_30_dataset.csv", "7749105_30_dataset.csv",
         "8000685_30_dataset.csv", "8173033_30_dataset.csv",
         "8258170_30_dataset.csv", "8530312_30_dataset.csv",
         "8686948_30_dataset.csv", "8692923_30_dataset.csv",
         "9106476_30_dataset.csv", "9961348_30_dataset.csv"]

test_files = ["9618981_30_dataset.csv"]

In [ ]:
from pathlib import Path

In [ ]:
df_list = []
for file in files:
    temp_df = pd.read_csv(file)

    file_name = Path(file).name

    try:
        # Розбиваємо назву файлу за нижнім підкресленням
        user_id = file_name.split('_')[0]
        temp_df['user_id'] = user_id
    except Exception as e:
        print(f"Помилка обробки файлу {file}: {e}")
        continue

    df_list.append(temp_df)

combined_df = pd.concat(df_list, ignore_index=True)

In [ ]:
test_df = pd.read_csv(test_files[0])
test_df['user_id'] = "9618981"

In [ ]:
test_df = test_df_orig.copy()

features = ['rr_norm_diff', 'rr_delta', 'mean_enmo', 'max_enmo', 'std_enmo']
label = "sleep_stage"
WINDOW_SIZE = 10  # 5 хвилин контексту (10 епох по 30 сек)

In [ ]:
test_df

,epoch_id,sleep_stage,mean_rr,std_rr,rr_norm_diff,rr_delta,mean_enmo,max_enmo,std_enmo,user_id
0,0,0,1068.493911,18.542456,0.000000,0.000000,0.000628,0.109906,0.006692,9618981
1,1,0,1050.567930,37.910012,-17.029682,-17.925981,0.000000,0.000000,0.000000,9618981
2,2,0,1036.010601,22.152664,-30.007660,-14.557329,0.000000,0.000000,0.000000,9618981
3,3,0,1051.250411,2.174646,-14.029458,15.239810,0.000008,0.004718,0.000163,9618981
4,4,0,955.821043,46.966309,-103.985884,-95.429368,0.009636,0.817591,0.055043,9618981
...,...,...,...,...,...,...,...,...,...,...
940,940,2,971.457787,11.598951,-5.403910,-9.114442,0.000818,0.005197,0.000909,9618981
941,941,2,959.216666,7.239310,-16.762780,-12.241121,0.000828,0.005562,0.000967,9618981
942,942,2,965.685522,13.865518,-9.779227,6.468856,0.000783,0.006554,0.000961,9618981
943,943,2,967.574277,2.405891,-7.495949,1.888755,0.000811,0.005940,0.000964,9618981


**для 5ти фаз**

In [ ]:
# лейбли сну 5->4
combined_df[label] = combined_df[label].replace({5: 4})
test_df[label] = test_df[label].replace({5: 4})

In [ ]:
#кастиль, перевірити!
combined_df = combined_df[combined_df[label] >= 0]
test_df = test_df[test_df[label] >= 0]

In [ ]:
def create_sequences(df, window_size):
    X, y = [], []
    # по юзерах
    for user_id, group in df.groupby('user_id'):
        data_x = group[features].values
        data_y = group[label].values

        for i in range(len(group) - window_size + 1):
            X.append(data_x[i : i + window_size])
            y.append(data_y[i + window_size - 1])

    return np.array(X), np.array(y)

In [ ]:
print("Нарізаємо вікна...")
X_train, Y_train = create_sequences(combined_df, WINDOW_SIZE)
X_test, Y_test = create_sequences(test_df, WINDOW_SIZE)

print(f"Форма X_train: {X_train.shape}")
print(f"Форма Y_train: {Y_train.shape}")

Нарізаємо вікна...
Форма X_train: (24339, 10, 5)
Форма Y_train: (24339,)


In [ ]:
#балансування
classes = np.unique(Y_train)
class_weights_array = compute_class_weight(class_weight='balanced', classes=classes, y=Y_train)
class_weights_dict = {cls: weight for cls, weight in zip(classes, class_weights_array)}

# 1D-CNN
model = Sequential([
    # Перший згортковий шар: вивчає мікро-патерни (наприклад, стрибки в межах 1-2 хвилин)
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(WINDOW_SIZE, len(features))),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),

    # Другий згортковий шар: вивчає макро-патерни
    Conv1D(filters=128, kernel_size=3, activation='relu'),
    BatchNormalization(),

    # Витягуємо дані в один вектор
    Flatten(),

    # Класифікатор
    Dense(64, activation='relu'),
    Dropout(0.5), # Захист від перенавчання
    Dense(5, activation='softmax') # 5 виходів (по одному на кожну фазу сну)
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

# 5. ТРЕНУВАННЯ
history = model.fit(
    X_train, Y_train,
    epochs=30,
    batch_size=64,
    class_weight=class_weights_dict,
    validation_split=0.1, # Відкушуємо 10% на валідацію, щоб бачити, чи немає перенавчання
    verbose=1
)



/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_13"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_24 (Conv1D)              │ (None, 8, 64)          │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_24          │ (None, 8, 64)          │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_12 (MaxPooling1D) │ (None, 4, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_25 (Conv1D)              │ (None, 2, 128)         │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_25          │ (None, 2, 128)         │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_12 (Flatten)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 43,269 (169.02 KB)

 Trainable params: 42,885 (167.52 KB)

 Non-trainable params: 384 (1.50 KB)

Epoch 1/30
343/343 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.3459 - loss: 1.4991 - val_accuracy: 0.3533 - val_loss: 1.4271
Epoch 2/30
343/343 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.3702 - loss: 1.3151 - val_accuracy: 0.3841 - val_loss: 1.3856
Epoch 3/30
343/343 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.3832 - loss: 1.2808 - val_accuracy: 0.3615 - val_loss: 1.4002
Epoch 4/30
343/343 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.3837 - loss: 1.2646 - val_accuracy: 0.3402 - val_loss: 1.4461
Epoch 5/30
343/343 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.3836 - loss: 1.2440 - val_accuracy: 0.3451 - val_loss: 1.4108
Epoch 6/30
343/343 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.3960 - loss: 1.2354 - val_accuracy: 0.3533 - val_loss: 1.4068
Epoch 7/30
343/343 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.4031 - loss: 1.2153 - val_accuracy: 0.4067 - val_loss: 1.3170
Epoch 8/30
343/343 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.4123 - loss: 1.1920 - val_accuracy: 0.

In [ ]:
target_names_5 = ['Wake', 'N1', 'N2', 'N3', 'REM']

# 1. Малюємо графіки
plot_learning_curves(history, model_name="1D-CNN (5 фаз)")

# 2. Оцінка на ТРЕНУВАЛЬНІЙ вибірці
print("\n==================================================")
print("   МЕТРИКИ 1D-CNN НА ТРЕНУВАЛЬНІЙ ВИБІРЦІ (TRAIN) ")
print("==================================================")
y_train_pred_probs = model.predict(X_train, verbose=0)
y_train_pred = np.argmax(y_train_pred_probs, axis=1)
print(f"Загальна точність (Accuracy) Train: {accuracy_score(Y_train, y_train_pred):.3f}")

# ВИПРАВЛЕНО: y_train_pred замість y_train_pred_cnn, та додано 4-й індекс у labels
print(classification_report(Y_train, y_train_pred, labels=[0, 1, 2, 3, 4], target_names=target_names_5, zero_division=0))

# 3. Оцінка на ТЕСТОВІЙ вибірці (Test)
print("\n==================================================")
print("   МЕТРИКИ 1D-CNN НА ТЕСТОВІЙ ВИБІРЦІ (TEST)      ")
print("==================================================")
y_test_pred_probs = model.predict(X_test, verbose=0)
y_test_pred = np.argmax(y_test_pred_probs, axis=1)
print(f"Загальна точність (Accuracy) Test: {accuracy_score(Y_test, y_test_pred):.3f}")

# ВИПРАВЛЕНО: додано 4-й індекс у labels
print(classification_report(Y_test, y_test_pred, labels=[0, 1, 2, 3, 4], target_names=target_names_5, zero_division=0))


Оцінка на тестовому юзері:
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
Загальна точність (Accuracy): 0.365
              precision    recall  f1-score   support

        Wake       0.62      0.36      0.46        85
          N1       0.32      0.24      0.27       165
          N2       0.44      0.20      0.27       348
          N3       0.07      0.88      0.13        25
         REM       0.62      0.58      0.60       313

    accuracy                           0.37       936
   macro avg       0.42      0.45      0.35       936
weighted avg       0.49      0.37      0.40       936



**для 4х фаз**

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight


In [ ]:
test_df = test_df_orig.copy()

WINDOW_SIZE = 10
features = ['rr_norm_diff', 'rr_delta', 'mean_enmo', 'max_enmo', 'std_enmo']
label = "sleep_stage"

mapping_dict = {
    0: 0, # Wake
    1: 1, # N1 -> Light
    2: 1, # N2 -> Light
    3: 2, # N3 -> Deep
    4: 3, # REM
    5: 3  # REM (оригінальний лейбл)
}


combined_df[label] = combined_df[label].replace(mapping_dict)
test_df[label] = test_df[label].replace(mapping_dict)

#проскакувало -1
combined_df = combined_df[combined_df[label] >= 0]
test_df = test_df[test_df[label] >= 0]



In [ ]:

# ГЕНЕРАЦІЯ ВІКОН З НАКЛАДАННЯМ
def create_sliding_sequences(df, window_size):
    X, y = [], []
    for user_id, group in df.groupby('user_id'):
        data_x = group[features].values
        data_y = group[label].values

        for i in range(len(group) - window_size + 1):
            X.append(data_x[i : i + window_size])
            y.append(data_y[i + window_size - 1])

    return np.array(X), np.array(y)

print("Нарізаємо Sliding вікна для CNN...")
X_train_cnn, Y_train_cnn = create_sliding_sequences(combined_df, WINDOW_SIZE)
X_test_cnn, Y_test_cnn = create_sliding_sequences(test_df, WINDOW_SIZE)

print(f"Форма X_train_cnn (має бути ~4500): {X_train_cnn.shape}")



Нарізаємо Sliding вікна для CNN...
Форма X_train_cnn (має бути ~4500): (24339, 10, 5)


In [ ]:

# БАЛАНСУВАННЯ
classes = np.unique(Y_train_cnn)
class_weights_array = compute_class_weight(class_weight='balanced', classes=classes, y=Y_train_cnn)
class_weights_dict = {cls: weight for cls, weight in zip(classes, class_weights_array)}


In [ ]:


# 1D-CNN
model_cnn = Sequential([
    tf.keras.Input(shape=(WINDOW_SIZE, len(features))),

    Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Dropout(0.1),

    Conv1D(filters=128, kernel_size=3, activation='relu', padding='same'),
    BatchNormalization(),
    Dropout(0.1),

    Flatten(),

    Dense(64, activation='relu'),
    Dropout(0.5),

    Dense(4, activation='softmax')
])

model_cnn.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])


# ТРЕНУВАННЯ (З ранньою зупинкою)
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print("\nПочинаємо тренування 1D-CNN...")
history = model_cnn.fit(
    X_train_cnn, Y_train_cnn,
    epochs=30,
    batch_size=64,
    class_weight=class_weights_dict,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

target_names = ['Wake', 'Light Sleep', 'Deep Sleep', 'REM']

# графіки
plot_learning_curves(history, model_name="1D-CNN")

# ТРЕНУВАЛЬНА
print("\n==================================================")
print("   МЕТРИКИ 1D-CNN НА ТРЕНУВАЛЬНІЙ ВИБІРЦІ (TRAIN) ")
print("==================================================")
y_train_pred_probs_cnn = model_cnn.predict(X_train_cnn, verbose=0)
y_train_pred_cnn = np.argmax(y_train_pred_probs_cnn, axis=1)
print(f"Загальна точність (Accuracy) Train: {accuracy_score(Y_train_cnn, y_train_pred_cnn):.3f}")
print(classification_report(Y_train_cnn, y_train_pred_cnn, labels=[0, 1, 2, 3], target_names=target_names, zero_division=0))

# ТЕСТОВА
print("\n==================================================")
print("   МЕТРИКИ 1D-CNN НА ТЕСТОВІЙ ВИБІРЦІ (TEST)      ")
print("==================================================")
y_test_pred_probs_cnn = model_cnn.predict(X_test_cnn, verbose=0)
y_test_pred_cnn = np.argmax(y_test_pred_probs_cnn, axis=1)
print(f"Загальна точність (Accuracy) Test: {accuracy_score(Y_test_cnn, y_test_pred_cnn):.3f}")
print(classification_report(Y_test_cnn, y_test_pred_cnn, labels=[0, 1, 2, 3], target_names=target_names, zero_division=0))




Починаємо тренування 1D-CNN...
Epoch 1/30
343/343 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.4037 - loss: 1.2210 - val_accuracy: 0.3809 - val_loss: 1.2892
Epoch 2/30
343/343 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.4152 - loss: 1.0718 - val_accuracy: 0.3578 - val_loss: 1.2555
Epoch 3/30
343/343 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.4116 - loss: 1.0438 - val_accuracy: 0.3817 - val_loss: 1.2076
Epoch 4/30
343/343 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.4193 - loss: 1.0276 - val_accuracy: 0.3776 - val_loss: 1.2302
Epoch 5/30
343/343 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.4199 - loss: 1.0152 - val_accuracy: 0.3587 - val_loss: 1.2679
Epoch 6/30
343/343 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.4287 - loss: 1.0065 - val_accuracy: 0.4380 - val_loss: 1.2212
Epoch 7/30
343/343 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.4367 - loss: 0.9917 - val_accuracy: 0.3891 - val_loss: 1.2715
Epoch 8/30
343/343 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.4335

LSTM

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight


In [ ]:

WINDOW_SIZE = 6
features = ['rr_norm_diff', 'rr_delta', 'mean_enmo', 'max_enmo', 'std_enmo']


In [ ]:
df_list = []
for file in files:
    temp_df = pd.read_csv(file)

    file_name = Path(file).name

    try:
        # Розбиваємо назву файлу за нижнім підкресленням
        user_id = file_name.split('_')[0]
        temp_df['user_id'] = user_id
    except Exception as e:
        print(f"Помилка обробки файлу {file}: {e}")
        continue

    df_list.append(temp_df)

combined_df = pd.concat(df_list, ignore_index=True)

In [ ]:
test_df = pd.read_csv(test_files[0])
test_df['user_id'] = "9618981"

In [ ]:
# МАПІНГ КЛАСІВ (щоб їх точно було від 0 до 3)
mapping_dict = {
    0: 0, # Wake
    1: 1, # N1 -> Light
    2: 1, # N2 -> Light
    3: 2, # N3 -> Deep
    4: 3, # REM (якщо ти вже робила заміну 5->4)
    5: 3  # REM (оригінальний лейбл)
}

combined_df['sleep_stage'] = combined_df['sleep_stage'].replace(mapping_dict)
test_df['sleep_stage'] = test_df['sleep_stage'].replace(mapping_dict)


In [ ]:
combined_df = combined_df[combined_df[label] >= 0]
test_df = test_df[test_df[label] >= 0]

In [ ]:

# БЕЗ НАКЛАДАННЯ
def create_tumbling_sequences(df, window_size):
    X, y = [], []
    for user_id, group in df.groupby('user_id'):
        data_x = group[features].values
        data_y = group['sleep_stage'].values

        # Крок дорівнює window_size (Tumbling Window)
        for i in range(0, len(group) - window_size + 1, window_size):
            X.append(data_x[i : i + window_size])
            y.append(data_y[i + window_size - 1])

    return np.array(X), np.array(y)

In [ ]:
print("Нарізаємо LSTM вікна без накладання...")
X_train_lstm, Y_train_lstm = create_tumbling_sequences(combined_df, WINDOW_SIZE)
X_test_lstm, Y_test_lstm = create_tumbling_sequences(test_df, WINDOW_SIZE)

print(f"Нова форма X_train: {X_train_lstm.shape}")

# 2. ВАГИ КЛАСІВ
classes = np.unique(Y_train_lstm)
weights_arr = compute_class_weight(class_weight='balanced', classes=classes, y=Y_train_lstm)
class_weights_dict = {cls: weight for cls, weight in zip(classes, weights_arr)}


Нарізаємо LSTM вікна без накладання...
Нова форма X_train: (4092, 6, 5)


In [ ]:

# LSTM
model_lstm = Sequential([
    LSTM(64, return_sequences=True, input_shape=(WINDOW_SIZE, len(features))),
    Dropout(0.3),

    LSTM(32),
    Dropout(0.3),

    Dense(32, activation='relu'),
    Dense(4, activation='softmax')
])

model_lstm.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
                   loss='sparse_categorical_crossentropy',
                   metrics=['accuracy'])

# !!!EarlyStopping
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print("\nПочинаємо тренування LSTM...")
history = model_lstm.fit(
    X_train_lstm, Y_train_lstm,
    epochs=40,
    batch_size=32,
    class_weight=class_weights_dict,
    validation_split=0.15,
    callbacks=[early_stop], # Зупинить навчання, якщо val_loss почне рости
    verbose=1
)

target_names = ['Wake', 'Light Sleep', 'Deep Sleep', 'REM']

#  графіки
plot_learning_curves(history, model_name="LSTM")

# ТРЕНУВАЛЬНА
print("\n==================================================")
print("   МЕТРИКИ LSTM НА ТРЕНУВАЛЬНІЙ ВИБІРЦІ (TRAIN)   ")
print("==================================================")
y_train_pred_probs_lstm = model_lstm.predict(X_train_lstm, verbose=0)
y_train_pred_lstm = np.argmax(y_train_pred_probs_lstm, axis=1)
print(f"Загальна точність (Accuracy) Train: {accuracy_score(Y_train_lstm, y_train_pred_lstm):.3f}")
print(classification_report(Y_train_lstm, y_train_pred_lstm, labels=[0, 1, 2, 3], target_names=target_names, zero_division=0))

# ТЕСТОВА
print("\n==================================================")
print("   МЕТРИКИ LSTM НА ТЕСТОВІЙ ВИБІРЦІ (TEST)        ")
print("==================================================")
y_test_pred_probs_lstm = model_lstm.predict(X_test_lstm, verbose=0)
y_test_pred_lstm = np.argmax(y_test_pred_probs_lstm, axis=1)
print(f"Загальна точність (Accuracy) Test: {accuracy_score(Y_test_lstm, y_test_pred_lstm):.3f}")
print(classification_report(Y_test_lstm, y_test_pred_lstm, labels=[0, 1, 2, 3], target_names=target_names, zero_division=0))



Починаємо тренування LSTM...
Epoch 1/40


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


109/109 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - accuracy: 0.4215 - loss: 1.3129 - val_accuracy: 0.4104 - val_loss: 1.3012
Epoch 2/40
109/109 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.3930 - loss: 1.1790 - val_accuracy: 0.4658 - val_loss: 1.2496
Epoch 3/40
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.4077 - loss: 1.1266 - val_accuracy: 0.3550 - val_loss: 1.2472
Epoch 4/40
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.3890 - loss: 1.0934 - val_accuracy: 0.3909 - val_loss: 1.2030
Epoch 5/40
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.4117 - loss: 1.0815 - val_accuracy: 0.3925 - val_loss: 1.2062
Epoch 6/40
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.4077 - loss: 1.0739 - val_accuracy: 0.4104 - val_loss: 1.1896
Epoch 7/40
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.3999 - loss: 1.0638 - val_accuracy: 0.3990 - val_loss: 1.2100
Epoch 8/40
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.4097 - loss: 1.0552 - val_accuracy: 0.369